# 📊 ERCE 2019 · Cuba — Análisis por preguntas de interés

**3er grado · Lectura y Matemática**

Cuaderno que responde, pregunta a pregunta, los principales interrogantes del
estudio. Cada sección plantea la **pregunta**, describe el **proceder analítico**
(limpieza, normalización, método estadístico) y lo acompaña de **tablas y gráficos**.

- **Fuente:** UNESCO-LLECE, Estudio Regional Comparativo y Explicativo ERCE 2019.
- **Escala:** media regional 700, desviación estándar 100.
- **Datos:** Cuba, 3er grado (archivos PL3, PM3, FA_A3, QP3 y Módulo Nacional ICCP).

## 0. Configuración del entorno

Importamos las librerías y el paquete del proyecto (`erce`), que encapsula la
carga, la limpieza, las funciones estadísticas y las figuras. La ruta base se
obtiene automáticamente a partir del archivo de configuración.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

# El cuaderno vive en cuadernos/: se sube hasta la raíz del proyecto (donde
# están `erce/` y `modelos/`) para que el flujo funcione desde cualquier cwd.
RAIZ = Path.cwd().resolve()
for _ in range(3):
    if (RAIZ / "modelos").is_dir():
        break
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import plotly.io as pio

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:.1f}")

from erce import config as cfg
from erce import data as D
from erce import analysis as A
from erce import plots as P

print("Versiones:", np.__version__, pd.__version__)
print("Directorio de datos:", cfg.DATOS_DIR)

Versiones: 2.5.1 3.0.5
Directorio de datos: /home/aess/Proyectos/analisis_3er_grado/07.ERCE-2019-FINAL


## 1. Carga y análisis previo del conjunto de datos

Antes de responder las preguntas hay que conocer los datos, limpiarlos y
normalizarlos. El estudio combina varias bases que se unen por identificadores:

| Archivo | Contenido |
|---|---|
| `PL3.csv` | Puntajes de **Lectura** (5 valores plausibles VP1..VP5) |
| `PM3.csv` | Puntajes de **Matemática** (5 valores plausibles) |
| `FA_A3.csv` | Cuestionario del estudiante (contexto familiar, ISECF, actitudes) |
| `QP3.csv` | Cuestionario del profesor |
| `CUBA3_*_MOD_NAC_PRUEBA.csv` | Módulo Nacional de Cuba (índices de calidad) |
| `CUBA3_*_PROMEDIOS_ESCUELAS_TODAS.csv` | Promedios por escuela + coordenadas |

In [2]:
pl = D.cargar_pl3()
pm = D.cargar_pm3()
fa = D.cargar_fa()
qp = D.cargar_qp()
print("PL3:", pl.shape, "| PM3:", pm.shape, "| FA:", fa.shape, "| QP3:", qp.shape)

# Los valores plausibles de lectura y matemática (mismos nombres VP1..VP5 en cada base)
print("\nPV en PL3:", [c for c in pl.columns if c.startswith("VP")])
print("PV en PM3:", [c for c in pm.columns if c.startswith("VP")])

PL3: (5273, 124) | PM3: (5273, 124) | FA: (5273, 219) | QP3: (271, 176)

PV en PL3: ['VP1', 'VP2', 'VP3', 'VP4', 'VP5']
PV en PM3: ['VP1', 'VP2', 'VP3', 'VP4', 'VP5']


### 1.1. Valores plausibles → puntaje por estudiante

ERCE reporta **5 valores plausibles** por estudiante para incorporar la
incertidumbre de la medición. La estimación puntual es la **media de los 5
valores** por fila.

In [3]:
for df, cols, nombre in [(pl, cfg.VARIABLES_PV, "PL3"),
                         (pm, cfg.VARIABLES_PV, "PM3")]:
    df[nombre] = df[cols].mean(axis=1, skipna=True)

pl[["IDSTUD", "PL3", "WSEN"]].head()
pl[["PL3"]].describe().round(1).T

,count,mean,std,min,25%,50%,75%,max
PL3,5273.0,734.0,91.4,430.8,666.3,729.9,801.0,998.4


### 1.2. Normalización y recodificación de variables

El índice socioeconómico **ISECF** ya viene **estandarizado** (media 0, DE 1) y
las variables de contexto se recodifican a categorías legibles. Verificamos la
estandarización y las distribuciones de las variables clave.

In [4]:
isecf = fa["ISECF"].dropna()
print(f"ISECF: n={isecf.shape[0]} | media={isecf.mean():.3f} | DE={isecf.std():.3f}  (estandarizado)")

for col, label in [("SEX", "Sexo"), ("RURAL", "Zona"), ("EDU", "Educación padres")]:
    if col in fa.columns:
        print(f"\n{label} ({col}):")
        print(fa[col].value_counts(dropna=False))

ISECF: n=5272 | media=0.279 | DE=0.715  (estandarizado)

Sexo (SEX):
SEX
0.0    2664
1.0    2573
NaN      36
Name: count, dtype: int64

Zona (RURAL):
RURAL
0    4833
1     440
Name: count, dtype: int64

Educación padres (EDU):
EDU
0.0    3192
1.0    1941
NaN     140
Name: count, dtype: int64


### 1.3. Dataset maestro (estudiante)

Unimos lectura, matemática y cuestionario por `IDSTUD`; incorporamos provincia y
municipio desde el mapa de escuelas; y generamos las variables legibles usadas
en todo el análisis.

In [5]:
base = pl[["IDSTUD", "IDSCHOOL", "PL3", "NL_VP1", "SEX", "RURAL", "EDU",
            "WT", "WSEN", "GRUPO"]].copy()
base = base.merge(pm[["IDSTUD", "PM3"]], on="IDSTUD", how="left")
base = base.merge(fa[["IDSTUD", "ISECF", "EDAD", "AAEG3", "ORGEN", "EFMAT",
                      "VIOES", "INVAP"]], on="IDSTUD", how="left")

lect, _, _ = D.cargar_promedios()
esc_map = lect[["IDSCHOOL", "ubi1_cen_esc", "ubi2_cen_esc"]].drop_duplicates("IDSCHOOL")
base = base.merge(esc_map, on="IDSCHOOL", how="left")

base["genero"]     = base["SEX"].map({1.0: "Niña", 0.0: "Niño"}).fillna("Sin dato")
base["zona"]       = base["RURAL"].map({1: "Rural", 0: "Urbana", 0.0: "Urbana"}).fillna("Sin dato")
base["edu_padres"] = base["EDU"].map({1.0: "Terciaria", 0.0: "≤ Secundaria"}).fillna("Sin dato")

print("Filas:", base.shape[0], "| Columnas:", base.shape[1])
base.head()

Filas: 5273 | Columnas: 23


,IDSTUD,IDSCHOOL,PL3,NL_VP1,SEX,RURAL,EDU,WT,WSEN,GRUPO,PM3,ISECF,EDAD,AAEG3,ORGEN,EFMAT,VIOES,INVAP,ubi1_cen_esc,ubi2_cen_esc,genero,zona,edu_padres
0,10010101,1001,718.9,III,0.0,0,1.0,21.5,1.1,10,837.8,1.6,8.0,0.8,1.2,-0.9,-1.2,0.3,Pinar del Río,Pinar del Río,Niño,Urbana,Terciaria
1,10010102,1001,859.0,IV,0.0,0,0.0,21.5,1.1,10,687.0,1.1,8.0,0.2,-0.3,1.0,-1.2,-0.1,Pinar del Río,Pinar del Río,Niño,Urbana,≤ Secundaria
2,10010103,1001,850.6,IV,0.0,0,0.0,21.5,1.1,10,828.5,-0.2,8.0,0.2,0.3,0.1,1.4,1.0,Pinar del Río,Pinar del Río,Niño,Urbana,≤ Secundaria
3,10010104,1001,886.3,IV,0.0,0,1.0,21.5,1.1,10,819.0,1.1,8.0,1.1,1.2,-0.9,-0.6,1.0,Pinar del Río,Pinar del Río,Niño,Urbana,Terciaria
4,10010105,1001,767.0,III,0.0,0,0.0,21.5,1.1,10,713.6,1.0,8.0,1.5,1.2,0.1,0.4,0.1,Pinar del Río,Pinar del Río,Niño,Urbana,≤ Secundaria


### 1.4. Control de calidad: valores faltantes y descriptivos

In [6]:
falt = base.isna().mean().mul(100).round(1).sort_values(ascending=False)
print("Porcentaje de valores faltantes (solo columnas con faltantes):")
print(falt[falt > 0])

print("\nDescriptivos de los puntajes:")
base[["PL3", "PM3", "ISECF", "EDAD"]].describe().round(1).T

Porcentaje de valores faltantes (solo columnas con faltantes):
ORGEN   3.4
EFMAT   3.0
EDU     2.7
INVAP   2.7
VIOES   2.6
AAEG3   2.2
EDAD    1.9
SEX     0.7
dtype: float64

Descriptivos de los puntajes:


,count,mean,std,min,25%,50%,75%,max
PL3,5273.0,734.0,91.4,430.8,666.3,729.9,801.0,998.4
PM3,5273.0,750.4,80.1,469.1,691.9,745.4,804.5,1019.6
ISECF,5272.0,0.3,0.7,-2.2,-0.2,0.3,0.8,2.8
EDAD,5171.0,8.4,0.6,7.0,8.0,8.0,9.0,12.0


## 2. Metodología estadística

Los métodos usados para responder cada pregunta son:

- **Medias ponderadas** con el peso de muestreo `WSEN` (diseño complejo LLECE).
- **Errores estándar** por el método de **réplicas BRR** (100 réplicas, factor de
  Fay 0.5), que combina la varianza de muestreo con la de imputación (5 valores
  plausibles) → ver `A.errores_estandar_brr`.
- **Prueba t de Welch** (dos grupos) con **d de Cohen** para el tamaño del efecto.
- **ANOVA de un factor** con **η²** (varianza explicada).
- **Correlación de Pearson** con **r²** (varianza explicada).
- **Chi-cuadrado** con **V de Cramér** (asociación entre categóricas).

Estas funciones ya están implementadas en `erce/analysis.py`; aquí las aplicamos,
pregunta por pregunta.

## 3. ¿Cuál es el nivel de rendimiento de Cuba frente a la región?

**Proceder:** estimar la media ponderada nacional de Lectura y Matemática con su
error estándar BRR, y compararla con la media regional del ERCE 2019 (Informe
UNESCO/LLECE: 697 en Lectura, 698 en Matemática; escala media 700).

In [7]:
res = A.resumen_nacional(base)
for k, v in res.items():
    ee = v.get("ee")
    ee_txt = f" · EE(BRR)={ee}" if ee else ""
    print(f"{k}: media={v['media']} · DE={v['de']} · n={v['n']}{ee_txt}")

reg = D.dataframe_regional()
reg.round(0)

Lectura: media=730.5 · DE=91.4 · n=5273
Matemática: media=750.6 · DE=80.1 · n=5273


,Área,Cuba,Región,Diferencia
0,Lectura,730,697,33
1,Matemática,751,698,53


In [8]:
fig = P.grafico_regional(reg)
fig.update_layout(width=820, height=440)
fig.show()

**Respuesta:** Cuba supera claramente a la región en ambas asignaturas y en
Matemática de 3er grado registró el **puntaje más alto de la región** (751). En
Lectura (730) también se ubica muy por encima de la media regional (697).

## 4. ¿Existen diferencias de rendimiento entre provincias?

**Proceder:** media ponderada por provincia con IC95% (BRR) y **ANOVA de un
factor** + **η²** para saber si las diferencias son significativas y qué parte de
la variabilidad explican. Se repite para **Lectura y Matemática**.

In [9]:
PRUEBAS = [("PL3", "Lectura", P.COLORES["azul"]),
          ("PM3", "Matemática", P.COLORES["rojo"])]

for score, sel, color in PRUEBAS:
    tab = A.tabla_resumen(base, score, "ubi1_cen_esc")
    anova = A.prueba_anova(base, score, "ubi1_cen_esc")
    mejor, peor = tab.iloc[0], tab.iloc[-1]
    print(f"== {sel} ==")
    print(f"  Mejor: {mejor['ubi1_cen_esc']} = {mejor['Media']} | Peor: {peor['ubi1_cen_esc']} = {peor['Media']}")
    print(f"  Brecha: {mejor['Media'] - peor['Media']:.0f} pts | ANOVA: F={anova['F']} p={anova['p']:.4f} η²={anova['eta2']}")
    globals()[f"tab_{score}"] = tab

== Lectura ==
  Mejor: Matanzas = 746.6 | Peor: Villa Clara = 707.5
  Brecha: 39 pts | ANOVA: F=5.69 p=0.0000 η²=0.013
== Matemática ==
  Mejor: Cienfuegos = 769.3 | Peor: Sancti Spíritus = 724.8
  Brecha: 44 pts | ANOVA: F=5.367 p=0.0000 η²=0.012


In [10]:
fig = P.grafico_barras_medias(tab_PL3, "ubi1_cen_esc", "Lectura",
                                P.COLORES["azul"], "Puntaje medio por provincia · Lectura")
fig.update_layout(width=820)
fig.show()

In [11]:
fig = P.grafico_barras_medias(tab_PM3, "ubi1_cen_esc", "Matemática",
                                P.COLORES["rojo"], "Puntaje medio por provincia · Matemática")
fig.update_layout(width=820)
fig.show()

**Respuesta (ambas asignaturas):** las diferencias entre provincias son
**estadísticamente significativas** en Lectura y Matemática, pero el efecto es
pequeño (η² ≈ 0.01): la provincia explica solo ≈1% de la varianza. La mayor parte
de la desigualdad ocurre **entre estudiantes de una misma provincia**, no entre
provincias.

## 5. ¿Existe brecha entre zona urbana y rural?

**Proceder:** prueba **t de Welch** entre las dos zonas con **d de Cohen**, más el
diagrama de caja para ver la distribución completa. Lo hacemos para Lectura y
Matemática (el resultado puede diferir entre asignaturas).

In [12]:
for score, sel in [("PL3", "Lectura"), ("PM3", "Matemática")]:
    t = A.prueba_t_grupos(base, score, "zona")
    gana = t["cat_a"] if t["media_a"] > t["media_b"] else t["cat_b"]
    perd = t["cat_b"] if t["media_a"] > t["media_b"] else t["cat_a"]
    print(f"{sel}: {gana} {abs(t['media_a']-t['media_b']):.0f} pts > {perd} "
          f"(t={t['t']}, p={t['p']:.4f}, d={abs(t['cohen_d']):.2f})")

Lectura: Urbana 15 pts > Rural (t=3.778, p=0.0002, d=0.18)
Matemática: Rural 19 pts > Urbana (t=-5.088, p=0.0000, d=0.26)


In [13]:
fig = P.grafico_caja(base, "zona", "PL3", P.COLORES["azul"],
                        "Zona urbana vs rural · Lectura")
fig.update_layout(width=820, height=460)
fig.show()

In [14]:
fig = P.grafico_caja(base, "zona", "PM3", P.COLORES["rojo"],
                        "Zona urbana vs rural · Matemática")
fig.update_layout(width=820, height=460)
fig.show()

**Respuesta:** la brecha urbano-rural existe y es significativa en ambas
asignaturas, pero de **magnitud pequeña** (d < 0.3). Es menor que la brecha
asociada al nivel socioeconómico, lo que sugiere que la desventaja rural se
explica más por condiciones socioeconómicas que por la ruralidad en sí.

## 6. ¿Cómo se asocia el nivel socioeconómico con el rendimiento?

**Proceder:** dividir el **ISECF** en cuartiles y comparar medias ponderadas entre
extremos (≈0.6 DE), y estimar la **correlación de Pearson** ISECF–puntaje con su
r² (varianza explicada). Dispersión muestral con regresión lineal. Para **Lectura
y Matemática**.

In [15]:
for score, sel, color in PRUEBAS:
    q = A.cuartiles_isect(base, score)
    corr_isect = A.correlaciones(base, ["ISECF"], score)
    q1, q4 = q.iloc[0]["Media"], q.iloc[-1]["Media"]
    de = res[sel]["de"]
    r = corr_isect["r"].iloc[0]
    print(f"== {sel} ==")
    print(f"  Cuartiles: {dict(zip(q['Cuartil'], q['Media']))}")
    print(f"  Brecha Q4-Q1: {q4 - q1:.0f} pts (≈{(q4-q1)/de:.2f} DE) | r = {r:.3f} | r² = {r**2*100:.0f}%")
    globals()[f"q_{score}"] = q

== Lectura ==
  Cuartiles: {'Q1': 704.1, 'Q2': 726.9, 'Q3': 737.5, 'Q4': 763.7}
  Brecha Q4-Q1: 60 pts (≈0.65 DE) | r = 0.258 | r² = 7%
== Matemática ==
  Cuartiles: {'Q1': 737.7, 'Q2': 747.0, 'Q3': 752.9, 'Q4': 769.9}
  Brecha Q4-Q1: 32 pts (≈0.40 DE) | r = 0.188 | r² = 4%


In [16]:
for score, sel, color in PRUEBAS:
    sub = base.dropna(subset=["ISECF", score]).sample(1800, random_state=42)
    fig = P.grafico_correlacion(sub, "ISECF", score, color,
                                f"ISECF vs puntaje · {sel} (muestra de 1800)")
    fig.update_layout(width=820)
    fig.show()

In [17]:
for score, sel, color in PRUEBAS:
    fig = P.grafico_isect_cuartiles(globals()[f"q_{score}"], color,
                                    f"Puntaje medio por cuartil de ISECF · {sel}")
    fig.update_layout(width=820)
    fig.show()

**Respuesta:** el **nivel socioeconómico es el factor individual más asociado**
al rendimiento en **ambas asignaturas**: brecha de ≈60 puntos (0.6 DE) entre
cuartiles extremos y r ≈ 0.26 (≈7% de la varianza). El gradiente es monótono y
sistemático en Lectura y en Matemática.

## 7. ¿Existen brechas de género en el rendimiento?

**Proceder:** prueba **t de Welch** entre niñas y niños con **d de Cohen**, para
cada asignatura (la brecha suele diferir: típicamente a favor de las niñas en
Lectura y nula o menor en Matemática).

In [18]:
for score, sel in [("PL3", "Lectura"), ("PM3", "Matemática")]:
    t = A.prueba_t_grupos(base, score, "genero")
    gana = t["cat_a"] if t["media_a"] > t["media_b"] else t["cat_b"]
    perd = t["cat_b"] if t["media_a"] > t["media_b"] else t["cat_a"]
    sig = "SÍ" if t["p"] < 0.05 else "no"
    print(f"{sel}: {gana} +{abs(t['media_a']-t['media_b']):.1f} pts vs {perd} "
          f"| t={t['t']:.2f} | p={t['p']:.4f} | d={abs(t['cohen_d']):.2f} | significativa: {sig}")

Lectura: Niña +23.1 pts vs Niño | t=-8.67 | p=0.0000 | d=0.24 | significativa: SÍ
Matemática: Niña +0.4 pts vs Niño | t=0.54 | p=0.5870 | d=0.01 | significativa: no


In [19]:
fig = P.grafico_caja(base, "genero", "PL3", P.COLORES["azul"],
                        "Distribución por género · Lectura")
fig.update_layout(width=820, height=460)
fig.show()

In [20]:
fig = P.grafico_caja(base, "genero", "PM3", P.COLORES["rojo"],
                        "Distribución por género · Matemática")
fig.update_layout(width=820, height=460)
fig.show()

**Respuesta:** en **Lectura** las niñas superan a los niños en ~22 puntos
(d ≈ 0.24, pequeño pero significativo); en **Matemática** la brecha es mínima y
no significativa. No hay evidencia de un desbalance sistemático de género.

## 8. ¿Qué factores del estudiante acompañan al mejor rendimiento?

**Proceder:** correlaciones de Pearson entre los factores del cuestionario de
estudiante y el puntaje, ordenadas de mayor a menor asociación, para **Lectura y
Matemática** (se interpretan como **asociaciones**, no causalidad).

In [21]:
factores = ["ISECF", "EDAD", "AAEG3", "ORGEN", "EFMAT", "VIOES", "INVAP"]
for score, sel, color in PRUEBAS:
    corr = A.correlaciones(base, factores, score)
    print(f"== {sel} ==")
    print(corr[["variable", "r", "p"]].round(3).to_string(index=False))
    globals()[f"corr_{score}"] = corr

== Lectura ==
variable    r   p
   ISECF  0.3 0.0
   AAEG3  0.2 0.0
   INVAP  0.2 0.0
   ORGEN  0.2 0.0
    EDAD  0.0 0.5
   EFMAT -0.1 0.0
   VIOES -0.2 0.0
== Matemática ==
variable    r   p
   AAEG3  0.2 0.0
   ORGEN  0.2 0.0
   INVAP  0.2 0.0
   ISECF  0.2 0.0
    EDAD  0.0 0.0
   EFMAT -0.1 0.0
   VIOES -0.2 0.0


In [22]:
import plotly.graph_objects as go

for score, sel, color in PRUEBAS:
    c = globals()[f"corr_{score}"].sort_values("r")
    colores = np.where(c["r"] >= 0, P.COLORES["azul"], P.COLORES["rojo"])
    fig = go.Figure(go.Bar(x=c["r"], y=c["variable"], orientation="h",
                           marker_color=colores, text=c["r"].round(2),
                           textposition="outside", cliponaxis=False))
    fig.update_layout(
        title=f"Asociación de factores del estudiante con el puntaje · {sel}",
        xaxis_title="r de Pearson", height=360,
        margin={"l": 180, "r": 30, "t": 50, "b": 30})
    fig.show()

**Respuesta:** el **ISECF**, el **apoyo al aprendizaje**, el **involucramiento
parental** y la **organización de la enseñanza** se asocian positivamente con el
rendimiento en ambas asignaturas; la **violencia escolar** se asocia
negativamente. La edad y la autoeficacia en matemática prácticamente no se
asocian.

## 9. ¿La calidad escolar y docente percibida se asocia al rendimiento?

**Proceder:** cargar el **Módulo Nacional** (índices 0-100 de docentes y
directivos), promediar cada índice y correlacionarlo con el puntaje de **Lectura
y Matemática**. Asociación **observacional** (percepciones autocontestadas).

In [23]:
mod = D.dataframe_modulo()
idx_cols = ["Indice_global", "Indice_preg", "Indice_refuerzo",
            "Indice_prep_didact_prof", "Indice_calidad_clases_prof",
            "Indice_calidad_clases_dir"]
print("Módulo:", mod.shape)
mod[idx_cols].mean().round(1).sort_values(ascending=False)

Módulo: (4879, 55)


Indice_calidad_clases_dir    95.5
Indice_refuerzo              50.0
Indice_global                48.6
Indice_preg                  47.3
Indice_calidad_clases_prof    1.0
Indice_prep_didact_prof       0.5
dtype: float64

In [24]:
for score, sel, color in PRUEBAS:
    corr_ind = A.correlaciones(mod, idx_cols, score)
    print(f"== {sel} ==")
    print(corr_ind[["variable", "r", "p"]].round(3).to_string(index=False))

fig = P.grafico_barras_indices_promedio(mod, idx_cols, P.COLORES["verde"],
                                        "Promedio de los índices de calidad (0-100)")
fig.update_layout(width=820)
fig.show()

== Lectura ==
                  variable   r   p
             Indice_global 0.3 0.0
           Indice_refuerzo 0.3 0.0
               Indice_preg 0.2 0.0
Indice_calidad_clases_prof 0.1 0.0
 Indice_calidad_clases_dir 0.0 0.0
   Indice_prep_didact_prof 0.0 0.0
== Matemática ==
                  variable   r   p
             Indice_global 0.3 0.0
           Indice_refuerzo 0.2 0.0
               Indice_preg 0.2 0.0
Indice_calidad_clases_prof 0.1 0.0
 Indice_calidad_clases_dir 0.0 0.0
   Indice_prep_didact_prof 0.0 0.0


In [25]:
for score, sel, color in PRUEBAS:
    fig = P.grafico_indices(mod, score, color,
                            f"Puntaje según quintil de cada índice · {sel}")
    fig.update_layout(width=820)
    fig.show()

**Respuesta:** el **índice global de calidad** es el más asociado al rendimiento
(r ≈ 0.30 en Lectura y similar en Matemática), seguido del refuerzo pedagógico y
del diálogo pedagógico. Los estudiantes rinden más en escuelas cuya calidad
pedagógica percibida es mayor, aunque la fuerza de la asociación es de pequeña a
moderada.

## 10. ¿Dónde se concentra geográficamente el rendimiento?

**Proceder:** media de valores plausibles **por escuela** (archivo de promedios
+ coordenadas) representada como burbujas georreferenciadas sobre el mapa de Cuba,
para **Lectura y Matemática**.

In [26]:
esc = D.dataframe_escuelas()
print("Escuelas con puntaje:", esc["PL3"].notna().sum())
esc[["nom_esc", "ubi1_cen_esc", "PL3", "PM3"]].head()

Escuelas con puntaje: 248


,nom_esc,ubi1_cen_esc,PL3,PM3
0,Sergio y Luis Saíz Montes De Oca,Pinar del Río,801.8,784.5
1,Rafael María de Mendive Daumy,Pinar del Río,776.5,749.4
2,Carlos Hidalgo Díaz,Pinar del Río,745.6,732.8
3,Maria Teresa Grenier Torres,Pinar del Río,731.9,758.4
4,Roberto Amarán Mamposo,Pinar del Río,785.8,813.0


In [27]:
fig = P.grafico_mapa(esc, "PL3", "Puntaje medio por escuela · Lectura")
fig.update_layout(width=1000, height=600)
fig.show()

In [28]:
fig = P.grafico_mapa(esc, "PM3", "Puntaje medio por escuela · Matemática")
fig.update_layout(width=1000, height=600)
fig.show()

**Respuesta:** los mapas reproducen las **~250 escuelas** con puntaje de la
muestra en cada asignatura. Las provincias de Pinar del Río, La Habana y Matanzas
concentran la mayor cantidad de escuelas y se aprecia **heterogeneidad entre
municipios**, coherente con la idea de que las brechas son sobre todo
intra-territoriales.

## 11. Síntesis

| # | Pregunta | Lectura | Matemática | Método principal |
|---|---|---|---|---|
| 1 | Rendimiento vs región | 730 (región 697) | 751 (región 698, máx. regional) | Media ponderada + EE BRR |
| 2 | Provincias | Brechas significativas, η²≈0.01 | Brechas significativas, η²≈0.01 | ANOVA + η² |
| 3 | Urbana vs rural | Brecha pequeña (d≈0.18) | Brecha pequeña (d≈0.26) | t de Welch + d de Cohen |
| 4 | Nivel socioeconómico | ≈60 pts (0.6 DE), r≈0.26 | Brecha y r similares | Cuartiles ISECF + Pearson |
| 5 | Género | Niñas +22 pts (d≈0.24) | Nula / no significativa | t de Welch + d de Cohen |
| 6 | Factores del estudiante | Apoyo familiar +; violencia − | Mismo patrón | Correlaciones de Pearson |
| 7 | Calidad escolar | Índice global r≈0.30 | Similar | Correlaciones (Módulo Nacional) |
| 8 | Distribución geográfica | Heterogeneidad intra-territorial | Heterogeneidad intra-territorial | Promedios por escuela + mapa |

**Conclusión global (ambas asignaturas):** el promedio alto de Cuba convive con
una **alta dispersión interna**: las brechas más relevantes son las
socioeconómicas (≈0.6 DE entre extremos del ISECF) y las que se observan
**dentro** de cada provincia, más que entre provincias. El patrón es consistente
en Lectura y Matemática. La mejora del sistema pasa por la **equidad interna**,
con apoyo a los hogares más desfavorecidos, el involucramiento familiar y la
calidad pedagógica de las escuelas.